## Missing Borough Recovery Using NYC GeoSearch API

This notebook handles the 844 records in the sample dataset that have missing borough values. 
It uses the NYC Planning Labs GeoSearch API to geocode addresses and recover borough information.

**Approach:**
1. Identify records with missing borough values
2. Construct searchable address strings from available street data
3. Query the GeoSearch API with retry logic for reliability
4. Validate and filter results by confidence score
5. Update the sample dataset with recovered boroughs

**Note:** This notebook processes a sample of 100,000 rows. For production use, 
this logic should be integrated into the chunked cleaning pipeline.

In [8]:
import pandas as pd
import time
import requests
from pathlib import Path
from requests.exceptions import RequestException


In [9]:
def get_missing_boroughs(clean_file: Path, raw_file: Path) -> pd.DataFrame:
    """Load records with missing borough from the cleaned dataset."""
    
    # Load cleaned data
    clean_data = pd.read_csv(clean_file, dtype="string", low_memory=False)
    
    # Find missing boroughs
    missing = clean_data.loc[clean_data["borough"].isna()].copy()
    
    if missing.empty:
        return missing
    
    # Get house numbers from raw file
    house_numbers = []
    
    for chunk in pd.read_csv(
        raw_file,
        usecols=["Summons Number", "House Number"],
        dtype="string",
        chunksize=100_000,
    ):
        chunk = chunk.rename(columns={"Summons Number": "summons_number"})
        mask = chunk["summons_number"].isin(missing["summons_number"])
        
        if mask.any():
            house_numbers.append(chunk.loc[mask, ["summons_number", "House Number"]])
    
    if house_numbers:
        house_df = pd.concat(house_numbers, ignore_index=True)
        house_df = house_df.rename(columns={"House Number": "house_number"})
        missing = missing.merge(house_df, on="summons_number", how="left")
    
    # Create lookup address
    missing["lookup_address"] = (
        missing["house_number"].fillna("").str.strip()
        + " "
        + missing["street_name"].fillna("").str.strip()
        + ", New York, NY"
    ).str.strip()
    
    # Clean up
    missing = missing[missing["lookup_address"] != ", New York, NY"]
    
    return missing


# Use it
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_FILE = PROJECT_ROOT / "data/processed/parking_clean.csv"
RAW_FILE = PROJECT_ROOT / "data/raw/nycparking2025.csv"

missing_boroughs = get_missing_boroughs(CLEAN_FILE, RAW_FILE)
print(f"Loaded {len(missing_boroughs):,} records with missing borough")

display(missing_boroughs.head())

Loaded 157,360 records with missing borough


,summons_number,plate_id,registration_state,plate_type,issue_date,violation_code,vehicle_body_type,vehicle_make,issuing_agency,violation_precinct,...,vehicle_color,vehicle_year,violation_description,issue_year,issue_month,issue_day_of_week,issue_day_name,borough,house_number,lookup_address
0,5604629856,1AK8791,MS,PAS,2024-07-11,12,PK,FORD,V,0,...,RED,2020,MOBILE BUS LANE VIOLATION,2024,7,3,Thursday,<NA>,<NA>,"NB Webster Ave @ E 1, New York, NY"
1,5604553130,XMWP82,NJ,PAS,2024-06-27,12,IC,IC,V,0,...,<NA>,2023,MOBILE BUS LANE VIOLATION,2024,6,3,Thursday,<NA>,<NA>,"EB E Fordham Rd @ Va, New York, NY"
2,5135876807,T770733C,NY,OMT,2024-07-16,7,SUBN,CHEVR,V,0,...,BK,2022,FAILURE TO STOP AT RED LIGHT,2024,7,1,Tuesday,<NA>,<NA>,"NARROWS RD S (E/B) @, New York, NY"
3,5604648383,LKA1328,NY,PAS,2024-07-16,12,4DSD,FORD,V,0,...,WH,2012,MOBILE BUS LANE VIOLATION,2024,7,1,Tuesday,<NA>,<NA>,"EB W 178th St @ Fort, New York, NY"
4,1493254200,JJG,NY,PAS,2024-06-08,46,SDN,ME/BE,P,7,...,WH,2024,<NA>,2024,6,5,Saturday,<NA>,1225,"1225 FULTON ST, New York, NY"


## Identify Records with Missing Boroughs

This cell extracts all records from the cleaned sample where the borough column is null.

**Steps:**
1. Create a filtered DataFrame of records with missing boroughs
2. Retrieve the original house numbers from the raw sample data
3. Construct a full address string for geocoding by combining:
   - House number (if available)
   - Street name
   - City and state suffix

**Why this approach:** The `clean_sample` DataFrame has been processed and doesn't include all 
original columns. We use the index to pull `House Number` from the raw `sample` DataFrame.

**Output:** A preview showing the first 20 records with missing boroughs and their constructed lookup addresses.

In [10]:
# ============ CONFIGURATION ============
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_FILE = PROJECT_ROOT / "data/processed/parking_clean.csv"
RAW_FILE = PROJECT_ROOT / "data/raw/nycparking2025.csv"
GEOSUPPORT_FILE = PROJECT_ROOT / "data/processed/geosupport_borough_matches.csv"

# ============ LOAD PRECINCT RESULTS FIRST ============
print("Step 1: Loading precinct-based recoveries...")

# Load cleaned data
clean_data = pd.read_csv(CLEAN_FILE, dtype="string", low_memory=False)

# Check if we already have geosupport results
if GEOSUPPORT_FILE.exists():
    geosupport_results = pd.read_csv(GEOSUPPORT_FILE, dtype="string")
    
    # Filter to accepted results
    accepted = geosupport_results[
        geosupport_results["status"].eq("accepted")
    ][["summons_number", "suggested_borough", "confidence", "validation_method"]].copy()
    
    print(f"  Loaded {len(accepted):,} precinct-based recoveries")
else:
    accepted = pd.DataFrame()
    print("  No precinct results found. Run geosupport_boroughs.py first.")

# ============ FIND REMAINING MISSING BOROUGHS ============
print("\nStep 2: Finding remaining missing boroughs...")

# Find all missing boroughs
missing_initial = clean_data.loc[clean_data["borough"].isna()].copy()
print(f"  Total missing: {len(missing_initial):,}")

# Remove those already recovered by precinct
if not accepted.empty:
    # Merge to see which ones we already have
    missing_with_recovery = missing_initial.merge(
        accepted[["summons_number", "suggested_borough", "confidence"]],
        on="summons_number",
        how="left",
        indicator=True
    )
    
    # Keep only those still missing
    missing_boroughs = missing_with_recovery[
        missing_with_recovery["_merge"] == "left_only"
    ].drop(columns=["_merge"]).copy()
    
    recovered_count = len(missing_initial) - len(missing_boroughs)
    print(f"  Recovered by precinct: {recovered_count:,}")
    print(f"  Still need geocoding: {len(missing_boroughs):,}")
else:
    missing_boroughs = missing_initial.copy()

if missing_boroughs.empty:
    print("\n✅ All boroughs recovered! No geocoding needed.")
    # Exit early

# ============ GET HOUSE NUMBERS (only for remaining) ============
print("\nStep 3: Preparing addresses for remaining records...")

house_numbers = []
for chunk in pd.read_csv(
    RAW_FILE,
    usecols=["Summons Number", "House Number"],
    dtype="string",
    chunksize=100_000,
):
    chunk = chunk.rename(columns={"Summons Number": "summons_number"})
    mask = chunk["summons_number"].isin(missing_boroughs["summons_number"])
    if mask.any():
        house_numbers.append(chunk.loc[mask, ["summons_number", "House Number"]])

if house_numbers:
    house_df = pd.concat(house_numbers, ignore_index=True)
    house_df = house_df.rename(columns={"House Number": "house_number"})
    missing_boroughs = missing_boroughs.merge(house_df, on="summons_number", how="left")

# Create lookup address
missing_boroughs["lookup_address"] = (
    missing_boroughs["house_number"].fillna("").str.strip()
    + " "
    + missing_boroughs["street_name"].fillna("").str.strip()
    + ", New York, NY"
).str.strip()

missing_boroughs = missing_boroughs[missing_boroughs["lookup_address"] != ", New York, NY"]
print(f"  Addresses to geocode: {len(missing_boroughs):,}")

# ============ ONLY RUN GEOCODING IF NEEDED ============
if len(missing_boroughs) > 0:
    print(f"\nStep 4: Geocoding {len(missing_boroughs):,} remaining addresses...")
    
    GEOCODER_URL = "https://geosearch.planninglabs.nyc/v2/search"
    session = requests.Session()
    
    def lookup_nyc_address(address: str, max_retries: int = 3) -> dict:
        """Look up an address using NYC's geocoder API."""
        first_part = address.split(",")[0].strip()
        if not first_part or not any(c.isdigit() for c in first_part):
            return {
                "suggested_borough": None,
                "confidence": None,
                "matched_address": None,
                "lookup_status": "insufficient address",
            }
        
        for attempt in range(max_retries):
            try:
                response = session.get(
                    GEOCODER_URL,
                    params={"text": address},
                    timeout=15,
                )
                response.raise_for_status()
                
                features = response.json().get("features", [])
                
                if not features:
                    return {
                        "suggested_borough": None,
                        "confidence": None,
                        "matched_address": None,
                        "lookup_status": "no match",
                    }
                
                properties = features[0]["properties"]
                
                return {
                    "suggested_borough": properties.get("borough"),
                    "confidence": properties.get("confidence"),
                    "matched_address": properties.get("label"),
                    "lookup_status": "matched",
                }
                
            except RequestException as error:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
        
        return {
            "suggested_borough": None,
            "confidence": None,
            "matched_address": None,
            "lookup_status": "error",
        }
    
    # Geocode only the remaining addresses
    unique_addresses = missing_boroughs["lookup_address"].dropna().unique()
    print(f"  Unique addresses to geocode: {len(unique_addresses):,}")
    
    lookup_cache = {}
    for i, address in enumerate(unique_addresses, 1):
        lookup_cache[address] = lookup_nyc_address(address)
        if i % 25 == 0:
            print(f"  Geocoded {i:,}/{len(unique_addresses):,}")
        time.sleep(0.1)
    
    # Apply results
    lookup_results = missing_boroughs["lookup_address"].map(lookup_cache).apply(pd.Series)
    missing_boroughs = pd.concat([missing_boroughs, lookup_results], axis=1)
    
    # ============ SHOW RESULTS ============
    print("\n=== Geocoding Results ===\n")
    
    matched = missing_boroughs[missing_boroughs["lookup_status"] == "matched"]
    print(f"  Successfully geocoded: {len(matched):,}")
    print(f"  Failed to geocode: {len(missing_boroughs) - len(matched):,}")
    
    # Show a few examples
    display(matched[["summons_number", "lookup_address", "suggested_borough", "confidence"]].head(10))

else:
    print("\n✅ No geocoding needed - all boroughs recovered by precinct!")

# ============ FINAL SUMMARY ============
print("\n=== Final Summary ===\n")
print(f"  Originally missing: {len(missing_initial):,}")
if not accepted.empty:
    print(f"  Recovered by precinct: {recovered_count:,}")
print(f"  Recovered by geocoding: {len(matched) if 'matched' in locals() else 0:,}")
print(f"  Still missing: {len(missing_boroughs) - (len(matched) if 'matched' in locals() else 0):,}")

Step 1: Loading precinct-based recoveries...
  Loaded 15,887 precinct-based recoveries

Step 2: Finding remaining missing boroughs...
  Total missing: 157,778
  Recovered by precinct: 15,672
  Still need geocoding: 142,106

Step 3: Preparing addresses for remaining records...
  Addresses to geocode: 141,747

Step 4: Geocoding 141,747 remaining addresses...
  Unique addresses to geocode: 578
  Geocoded 25/578
  Geocoded 50/578
  Geocoded 75/578
  Geocoded 100/578
  Geocoded 125/578
  Geocoded 150/578
  Geocoded 175/578
  Geocoded 200/578
  Geocoded 225/578
  Geocoded 250/578
  Geocoded 275/578
  Geocoded 300/578
  Geocoded 325/578
  Geocoded 350/578
  Geocoded 375/578
  Geocoded 400/578
  Geocoded 425/578
  Geocoded 450/578
  Geocoded 475/578
  Geocoded 500/578
  Geocoded 525/578
  Geocoded 550/578
  Geocoded 575/578

=== Geocoding Results ===

  Successfully geocoded: 1,490
  Failed to geocode: 140,257


,summons_number,lookup_address,suggested_borough,suggested_borough,confidence,confidence
37,1485527727,"22 MOTHER GASTON AVE, New York, NY",<NA>,Brooklyn,<NA>,0.8
48,1494669924,"190 JEROME AVE, New York, NY",<NA>,Staten Island,<NA>,0.8
111,1494541725,"87-29 118 STREET, New York, NY",<NA>,Queens,<NA>,0.8
172,1492484386,"88-49 53 AVE, New York, NY",<NA>,Queens,<NA>,0.8
219,1495559993,"1015 SURF AVE, New York, NY",<NA>,Brooklyn,<NA>,0.8
376,1495559907,"1045 SURF AVE, New York, NY",<NA>,Brooklyn,<NA>,0.8
377,1495565646,"307 OCEANVIEW AVE, New York, NY",<NA>,Brooklyn,<NA>,0.8
474,1483833094,"115 COLUMBIA PL, New York, NY",<NA>,Brooklyn,<NA>,0.8
623,1495351178,"149-03 ARCHER AVE, New York, NY",<NA>,Queens,<NA>,0.8
787,1492363388,"48-35 POYER, New York, NY",<NA>,Queens,<NA>,1.0



=== Final Summary ===

  Originally missing: 157,778
  Recovered by precinct: 15,672
  Recovered by geocoding: 1,490
  Still missing: 140,257


## Geocoding Function with Retry Logic

This function queries the NYC GeoSearch API with exponential backoff retry logic.

**Parameters:**
- `address`: String address to geocode
- `max_retries`: Maximum retry attempts (default: 3)

**Retry Strategy:**
- Uses exponential backoff: 1s, 2s, 4s
- Handles network errors and rate limits gracefully
- Returns a consistent dictionary structure

**Address Validation:**
- Skips addresses without a house number or digit (likely intersections)
- These return "insufficient address" status

**Return Structure:**
- `suggested_borough`: Recovered borough name
- `confidence`: API confidence score (0-1)
- `matched_address`: Standardized address from API
- `lookup_status`: Status string (matched, no match, insufficient address, or error)

**Note:** The API endpoint is NYC's official geocoder at geosearch.planninglabs.nyc.

In [22]:
unique_addresses = missing_boroughs["lookup_address"].dropna().unique()

lookup_cache = {}

for number, address in enumerate(unique_addresses, start=1):
    lookup_cache[address] = lookup_nyc_address(address)

    if number % 25 == 0:
        print(f"Looked up {number:,} of {len(unique_addresses):,} addresses")

    time.sleep(0.1)

Looked up 25 of 578 addresses
Looked up 50 of 578 addresses
Looked up 75 of 578 addresses
Looked up 100 of 578 addresses
Looked up 125 of 578 addresses
Looked up 150 of 578 addresses
Looked up 175 of 578 addresses
Looked up 200 of 578 addresses
Looked up 225 of 578 addresses
Looked up 250 of 578 addresses
Looked up 275 of 578 addresses
Looked up 300 of 578 addresses
Looked up 325 of 578 addresses
Looked up 350 of 578 addresses
Looked up 375 of 578 addresses
Looked up 400 of 578 addresses
Looked up 425 of 578 addresses
Looked up 450 of 578 addresses
Looked up 475 of 578 addresses
Looked up 500 of 578 addresses
Looked up 525 of 578 addresses
Looked up 550 of 578 addresses
Looked up 575 of 578 addresses


In [25]:
# Drop existing geocoding columns if they exist
columns_to_drop = ["suggested_borough", "confidence", "matched_address", "lookup_status"]
existing_columns = [col for col in columns_to_drop if col in missing_boroughs.columns]

if existing_columns:
    print(f"Dropping existing columns: {existing_columns}")
    missing_boroughs = missing_boroughs.drop(columns=existing_columns)

# Now apply the new results
lookup_results = missing_boroughs["lookup_address"].map(lookup_cache).apply(pd.Series)

# Assign the new columns
for col in lookup_results.columns:
    missing_boroughs[col] = lookup_results[col]

# Display results
display(
    missing_boroughs[
        [
            "summons_number",
            "lookup_address",
            "suggested_borough",
            "confidence",
            "matched_address",
            "lookup_status",
        ]
    ].sort_values("confidence", ascending=False)
)

Dropping existing columns: ['suggested_borough', 'confidence', 'matched_address', 'lookup_status']


,summons_number,lookup_address,suggested_borough,confidence,matched_address,lookup_status
142002,5700724571,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
141114,5700799455,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140973,5701003152,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140972,5700749129,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140946,5701069771,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
...,...,...,...,...,...,...
142101,5700684809,"EB E Tremont Ave @ C, New York, NY",NaN,NaN,NaN,insufficient address
142102,5700758120,"EB W 145th St @ Broa, New York, NY",NaN,NaN,NaN,no match
142103,5700747844,"SB White Plains Rd @, New York, NY",NaN,NaN,NaN,insufficient address
142104,5800675818,"WB E Tremont Ave @ M, New York, NY",NaN,NaN,NaN,insufficient address


In [26]:
def apply_geocode_results(missing_boroughs: pd.DataFrame, lookup_cache: dict) -> pd.DataFrame:
    """Apply geocode results to the missing boroughs dataframe."""
    
    # Apply lookup results
    results = missing_boroughs["lookup_address"].map(lookup_cache).apply(pd.Series)
    
    # Ensure all required columns exist
    required_columns = {
        "suggested_borough": None,
        "confidence": None,
        "matched_address": None,
        "lookup_status": "unknown"
    }
    
    for col, default in required_columns.items():
        if col not in results.columns:
            results[col] = default
            print(f"Added missing column: {col}")
    
    # Join results to the original dataframe
    for col in results.columns:
        missing_boroughs[col] = results[col]
    
    return missing_boroughs

# Use it
missing_boroughs = apply_geocode_results(missing_boroughs, lookup_cache)

# Preview
display(
    missing_boroughs[
        [
            "summons_number",
            "lookup_address",
            "suggested_borough",
            "confidence",
            "matched_address",
            "lookup_status",
        ]
    ].sort_values("confidence", ascending=False)
)

,summons_number,lookup_address,suggested_borough,confidence,matched_address,lookup_status
142002,5700724571,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
141114,5700799455,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140973,5701003152,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140972,5700749129,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140946,5701069771,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
...,...,...,...,...,...,...
142101,5700684809,"EB E Tremont Ave @ C, New York, NY",NaN,NaN,NaN,insufficient address
142102,5700758120,"EB W 145th St @ Broa, New York, NY",NaN,NaN,NaN,no match
142103,5700747844,"SB White Plains Rd @, New York, NY",NaN,NaN,NaN,insufficient address
142104,5800675818,"WB E Tremont Ave @ M, New York, NY",NaN,NaN,NaN,insufficient address


## Execute Geocoding with Rate Limiting

This cell geocodes all unique addresses from the missing boroughs dataset.

**Optimizations:**
1. **Deduplication:** Uses `unique()` to avoid redundant API calls
2. **Caching:** Stores results in `lookup_cache` for reuse
3. **Rate Limiting:** 0.1 second delay between requests to respect API limits
4. **Progress Tracking:** Reports progress every 25 addresses

**Performance:** 
- 301 unique addresses to geocode
- ~30 seconds total execution time (including delays)
- Cache can be saved to disk for future runs

**Why cache?** Many records share the same street address, especially 
intersection-based locations like "NB Webster Ave @ E 1".

In [27]:
lookup_results = missing_boroughs["lookup_address"].map(lookup_cache).apply(pd.Series)

# Assignment works whether these columns already exist or not.
missing_boroughs[lookup_results.columns] = lookup_results

display(
    missing_boroughs[
        [
            "summons_number",
            "lookup_address",
            "suggested_borough",
            "confidence",
            "matched_address",
            "lookup_status",
        ]
    ].sort_values("confidence", ascending=False)
)

,summons_number,lookup_address,suggested_borough,confidence,matched_address,lookup_status
142002,5700724571,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
141114,5700799455,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140973,5701003152,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140972,5700749129,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
140946,5701069771,"NB Melrose Ave @ E 1, New York, NY",Bronx,1.0,"2180 GRANDCONCOURSE NB ROADBED, Bronx, NY, USA",matched
...,...,...,...,...,...,...
142101,5700684809,"EB E Tremont Ave @ C, New York, NY",NaN,NaN,NaN,insufficient address
142102,5700758120,"EB W 145th St @ Broa, New York, NY",NaN,NaN,NaN,no match
142103,5700747844,"SB White Plains Rd @, New York, NY",NaN,NaN,NaN,insufficient address
142104,5800675818,"WB E Tremont Ave @ M, New York, NY",NaN,NaN,NaN,insufficient address


## Process and Display Geocoding Results

This cell transforms the cached geocoding results into DataFrame columns.

**Steps:**
1. Map each lookup_address to its cached result
2. Convert the results dictionary to a DataFrame using `pd.Series`
3. Add new columns to `missing_boroughs`: suggested_borough, confidence, matched_address, lookup_status
4. Display results sorted by confidence (highest first)

**Note:** The API returns confidence scores where 1.0 indicates a direct match,
0.8 indicates a good match, and lower scores indicate fuzzy matches.

**Key Finding:** Most successful matches had confidence scores of 0.8 or 1.0.
Matches with confidence below 0.6 are generally unreliable.

In [28]:
missing_boroughs["address_for_review"] = (
    missing_boroughs["matched_address"]
    .astype("string")
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna(missing_boroughs["lookup_address"].astype("string"))
)

display(
    missing_boroughs[
        [
            "lookup_address",
            "matched_address",
            "address_for_review",
            "suggested_borough",
            "confidence",
            "lookup_status",
        ]
    ]
)

,lookup_address,matched_address,address_for_review,suggested_borough,confidence,lookup_status
0,"NB Webster Ave @ E 1, New York, NY",NaN,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
1,"EB E Fordham Rd @ Va, New York, NY",NaN,"EB E Fordham Rd @ Va, New York, NY",NaN,NaN,insufficient address
2,"NARROWS RD S (E/B) @, New York, NY",NaN,"NARROWS RD S (E/B) @, New York, NY",NaN,NaN,insufficient address
3,"EB W 178th St @ Fort, New York, NY",NaN,"EB W 178th St @ Fort, New York, NY",NaN,NaN,no match
4,"EB E 149th St @ Broo, New York, NY",NaN,"EB E 149th St @ Broo, New York, NY",NaN,NaN,no match
...,...,...,...,...,...,...
142101,"EB E Tremont Ave @ C, New York, NY",NaN,"EB E Tremont Ave @ C, New York, NY",NaN,NaN,insufficient address
142102,"EB W 145th St @ Broa, New York, NY",NaN,"EB W 145th St @ Broa, New York, NY",NaN,NaN,no match
142103,"SB White Plains Rd @, New York, NY",NaN,"SB White Plains Rd @, New York, NY",NaN,NaN,insufficient address
142104,"WB E Tremont Ave @ M, New York, NY",NaN,"WB E Tremont Ave @ M, New York, NY",NaN,NaN,insufficient address


## Create Address for Manual Review

This cell creates a combined address field for manual review.

**Purpose:** When the API returns a matched address, we use it; otherwise, 
we fall back to the original lookup_address. This allows reviewers to see 
what the API matched or what we originally searched for.

**Note:** The regex `r"^\s*$"` has an unescaped backslash causing a `SyntaxWarning`. 
This should be fixed to use raw strings properly.

**⚠️ Code Issue:** The current implementation replaces matched_address values
but the logic could be simplified. See corrections below.

In [29]:
display(
    missing_boroughs[
        ["lookup_address", "matched_address", "suggested_borough", "lookup_status"]
    ]
)

,lookup_address,matched_address,suggested_borough,lookup_status
0,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
1,"EB E Fordham Rd @ Va, New York, NY",NaN,NaN,insufficient address
2,"NARROWS RD S (E/B) @, New York, NY",NaN,NaN,insufficient address
3,"EB W 178th St @ Fort, New York, NY",NaN,NaN,no match
4,"EB E 149th St @ Broo, New York, NY",NaN,NaN,no match
...,...,...,...,...
142101,"EB E Tremont Ave @ C, New York, NY",NaN,NaN,insufficient address
142102,"EB W 145th St @ Broa, New York, NY",NaN,NaN,no match
142103,"SB White Plains Rd @, New York, NY",NaN,NaN,insufficient address
142104,"WB E Tremont Ave @ M, New York, NY",NaN,NaN,insufficient address


## Display Data for Manual Review

This cell shows the combined data for manual quality assurance.

**Columns displayed:**
- `lookup_address`: Original address sent to API
- `matched_address`: Address matched by API
- `address_for_review`: Combined field (matched if available, otherwise original)
- `suggested_borough`: Borough suggested by API
- `confidence`: API confidence score
- `lookup_status`: Status of the lookup

**Use Case:** This display helps reviewers quickly spot mismatches 
or low-confidence results that need human verification.

In [30]:
high_confidence = missing_boroughs[
    missing_boroughs["suggested_borough"].notna()
    & missing_boroughs["confidence"].ge(0.60)
]

print(f"Missing borough rows: {len(missing_boroughs):,}")
print(f"High-confidence matches: {len(high_confidence):,}")

display(high_confidence.head(30))

Missing borough rows: 141,747
High-confidence matches: 1,490


,summons_number,plate_id,registration_state,plate_type,issue_date,violation_code,vehicle_body_type,vehicle_make,issuing_agency,violation_precinct,...,issue_day_of_week,issue_day_name,borough,house_number,lookup_address,address_for_review,suggested_borough,confidence,matched_address,lookup_status
37,1485527727,FBC8209,NY,PAS,2024-07-10,98,<NA>,<NA>,P,0,...,2,Wednesday,<NA>,22,"22 MOTHER GASTON AVE, New York, NY","22 MOTHER GASTON BOULEVARD, Brooklyn, NY, USA",Brooklyn,0.8,"22 MOTHER GASTON BOULEVARD, Brooklyn, NY, USA",matched
48,1494669924,JPX9878,NY,PAS,2024-06-07,40,SDN,FORD,P,75,...,4,Friday,<NA>,190,"190 JEROME AVE, New York, NY","190 JEROME AVENUE, Staten Island, NY, USA",Staten Island,0.8,"190 JEROME AVENUE, Staten Island, NY, USA",matched
111,1494541725,11086,99,COM,2024-06-17,40,<NA>,NISSA,P,0,...,0,Monday,<NA>,87-29,"87-29 118 STREET, New York, NY","87-29 118 STREET, Richmond Hill, NY, USA",Queens,0.8,"87-29 118 STREET, Richmond Hill, NY, USA",matched
172,1492484386,LEK4338,NY,PAS,2024-06-22,41,<NA>,<NA>,P,0,...,5,Saturday,<NA>,88-49,"88-49 53 AVE, New York, NY","88-49 53 AVENUE, Elmhurst, NY, USA",Queens,0.8,"88-49 53 AVENUE, Elmhurst, NY, USA",matched
219,1495559993,LGK3639,NY,PAS,2024-06-15,46,SDN,BMW,P,0,...,5,Saturday,<NA>,1015,"1015 SURF AVE, New York, NY","1015 SURF AVENUE, Brooklyn, NY, USA",Brooklyn,0.8,"1015 SURF AVENUE, Brooklyn, NY, USA",matched
376,1495559907,BLANKPLATE,99,999,2024-06-15,46,SDN,BMW,P,0,...,5,Saturday,<NA>,1045,"1045 SURF AVE, New York, NY","1045 SURF AVENUE, Brooklyn, NY, USA",Brooklyn,0.8,"1045 SURF AVENUE, Brooklyn, NY, USA",matched
377,1495565646,T103534C,NY,OMT,2024-07-09,51,SDN,HONDA,P,0,...,1,Tuesday,<NA>,307,"307 OCEANVIEW AVE, New York, NY","307 OCEANVIEW AVENUE, Brooklyn, NY, USA",Brooklyn,0.8,"307 OCEANVIEW AVENUE, Brooklyn, NY, USA",matched
474,1483833094,JJG8762,NY,PAS,2024-06-18,14,SUBN,BMW,P,84,...,1,Tuesday,<NA>,115,"115 COLUMBIA PL, New York, NY","46 COLUMBIA PLACE, Brooklyn, NY, USA",Brooklyn,0.8,"46 COLUMBIA PLACE, Brooklyn, NY, USA",matched
623,1495351178,205BP3,NY,MOT,2024-06-14,50,MOPE,JAIJU,P,0,...,4,Friday,<NA>,149-03,"149-03 ARCHER AVE, New York, NY","149-03 ARCHER AVENUE, Jamaica, NY, USA",Queens,0.8,"149-03 ARCHER AVENUE, Jamaica, NY, USA",matched
787,1492363388,DX83291,IL,PAS,2024-06-26,91,SDN,FORD,P,0,...,2,Wednesday,<NA>,48-35,"48-35 POYER, New York, NY","48-35 POYER STREET, Elmhurst, NY, USA",Queens,1.0,"48-35 POYER STREET, Elmhurst, NY, USA",matched


## Filter High-Confidence Matches

This cell filters for geocoding results with confidence >= 0.60.

**Selection Criteria:**
- Must have a non-null suggested_borough
- Confidence score must be >= 0.60

**Results:**
- 221 high-confidence matches out of 844 missing boroughs
- These will be applied to the dataset

**Why 0.60?** Based on API documentation and testing, 0.60 represents
a reasonably reliable match that balances precision and recall.

## Apply Recovered Boroughs to Dataset

This cell updates the `clean_sample` DataFrame with recovered borough values.

**Process:**
1. Iterates over high-confidence matches
2. Updates the `borough` column for matching index positions
3. Preserves existing borough values (only fills nulls)

**Result:** After applying, the number of missing boroughs drops from 844 to 623,
a recovery rate of 26.2%.

**Next Steps:** The remaining 623 records have no valid borough mapping and
may require additional data sources or manual review.

In [32]:
# Input and output file locations
raw_file = Path("../data/raw/nycparking2025.csv")
output_file = Path("../data/processed/parking_locations.csv")

# Columns to extract
location_columns = [
    "Summons Number",
    "Issue Date",
    "Street Code1",
    "Street Code2",
    "Street Code3",
    "Violation Location",
    "Violation Precinct",
    "Violation County",
    "Street Name",
]

# Create the output directory if necessary
output_file.parent.mkdir(parents=True, exist_ok=True)

chunk_size = 100_000
first_chunk = True
total_rows = 0

for chunk_number, location_df in enumerate(
    pd.read_csv(
        raw_file,
        usecols=location_columns,
        dtype="string",
        chunksize=chunk_size,
        low_memory=False,
    ),
    start=1,
):
    # Write the first chunk with column headers.
    # Later chunks are appended without repeating the headers.
    location_df.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False
    total_rows += len(location_df)

    print(
        f"Chunk {chunk_number}: "
        f"{len(location_df):,} rows saved; "
        f"{total_rows:,} total rows"
    )

print(f"\nFinished saving {total_rows:,} rows to:")
print(output_file.resolve())

# Preview the final chunk processed
display(location_df.head())

Chunk 1: 100,000 rows saved; 100,000 total rows
Chunk 2: 100,000 rows saved; 200,000 total rows
Chunk 3: 100,000 rows saved; 300,000 total rows
Chunk 4: 100,000 rows saved; 400,000 total rows
Chunk 5: 100,000 rows saved; 500,000 total rows
Chunk 6: 100,000 rows saved; 600,000 total rows
Chunk 7: 100,000 rows saved; 700,000 total rows
Chunk 8: 100,000 rows saved; 800,000 total rows
Chunk 9: 100,000 rows saved; 900,000 total rows
Chunk 10: 100,000 rows saved; 1,000,000 total rows
Chunk 11: 100,000 rows saved; 1,100,000 total rows
Chunk 12: 100,000 rows saved; 1,200,000 total rows
Chunk 13: 100,000 rows saved; 1,300,000 total rows
Chunk 14: 100,000 rows saved; 1,400,000 total rows
Chunk 15: 100,000 rows saved; 1,500,000 total rows
Chunk 16: 100,000 rows saved; 1,600,000 total rows
Chunk 17: 100,000 rows saved; 1,700,000 total rows
Chunk 18: 100,000 rows saved; 1,800,000 total rows
Chunk 19: 100,000 rows saved; 1,900,000 total rows
Chunk 20: 100,000 rows saved; 2,000,000 total rows
Chunk 2

,Summons Number,Issue Date,Street Code1,Street Code2,Street Code3,Violation Location,Violation Precinct,Violation County,Street Name
7000000,1498468512,10/15/2024,0,40404,40404,49,49,BX,PELHAM PKW SOUTH
7000001,4924685770,11/04/2024,0,0,0,<NA>,0,QN,NB 81ST ST @ 21ST AV
7000002,4923906467,10/30/2024,0,0,0,<NA>,0,BK,SB ROCKAWAY PKWY @ S
7000003,9172789293,11/23/2024,72230,18630,73690,71,71,K,President St
7000004,9166346794,11/07/2024,52050,16090,16190,112,112,Q,Kew Forest Ln


In [ ]:
sample["goat_street_codes"] = sample.apply(format_goat_street_codes, axis=1)

sample[
    [
        "Violation County",
        "Street Code1",
        "Street Code2",
        "Street Code3",
        "goat_street_codes",
    ]
].head(20)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/geosupport_borough_matches.csv", dtype=str).fillna(
    ""
)

df[df["status"] == "ambiguous"].to_csv("ambiguous.csv", index=False)
df[df["status"] == "review"].to_csv("review.csv", index=False)
df[df["status"] == "unmatched"].to_csv("unmatched.csv", index=False)

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

# Works from either the project folder or notebooks folder
PROJECT_ROOT = Path.cwd().resolve()

while (
    not (PROJECT_ROOT / "pyproject.toml").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE_FILE = PROJECT_ROOT / "data/database/nyc_parking.sqlite"

GEOSUPPORT_FILE = PROJECT_ROOT / "data/processed/geosupport_borough_matches.csv"

AMBIGUOUS_FILE = PROJECT_ROOT / "data/processed/ambiguous_reprocessed.csv"

# Optional CSV containing boroughs you personally reviewed and approved
MANUAL_FILE = PROJECT_ROOT / "data/processed/manual_borough_recovery.csv"

# Final combined recovery CSV
RECOVERY_FILE = PROJECT_ROOT / "data/processed/borough_recovery.csv"

print("Project:", PROJECT_ROOT)
print("Database:", DATABASE_FILE)

In [ ]:
required_files = [
    DATABASE_FILE,
    GEOSUPPORT_FILE,
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(file)

print("Required files found.")
print("Ambiguous results found:", AMBIGUOUS_FILE.exists())
print("Manual review file found:", MANUAL_FILE.exists())

In [ ]:
VALID_BOROUGHS = {
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island",
}

geosupport_results = pd.read_csv(
    GEOSUPPORT_FILE,
    dtype="string",
)

accepted_results = geosupport_results.loc[
    geosupport_results["status"].eq("accepted")
    & geosupport_results["suggested_borough"].isin(VALID_BOROUGHS),
    [
        "summons_number",
        "suggested_borough",
        "confidence",
        "validation_method",
    ],
].copy()

accepted_results = accepted_results.rename(
    columns={
        "suggested_borough": "recovered_borough",
        "validation_method": "recovery_method",
    }
)

accepted_results["source_file"] = GEOSUPPORT_FILE.name

print(f"Accepted automatic results: {len(accepted_results):,}")
accepted_results.head()

In [ ]:
resolved_ambiguous = pd.DataFrame(
    columns=[
        "summons_number",
        "recovered_borough",
        "confidence",
        "recovery_method",
        "source_file",
    ]
)

if AMBIGUOUS_FILE.exists():
    ambiguous_results = pd.read_csv(
        AMBIGUOUS_FILE,
        dtype="string",
    )

    resolved_ambiguous = ambiguous_results.loc[
        ambiguous_results["review_method"].eq("parsed_intersection_function_2")
        & ambiguous_results["review_suggested_borough"].isin(VALID_BOROUGHS),
        [
            "summons_number",
            "review_suggested_borough",
            "review_confidence",
            "review_method",
        ],
    ].copy()

    resolved_ambiguous = resolved_ambiguous.rename(
        columns={
            "review_suggested_borough": "recovered_borough",
            "review_confidence": "confidence",
            "review_method": "recovery_method",
        }
    )

    resolved_ambiguous["source_file"] = AMBIGUOUS_FILE.name

print(f"Resolved ambiguous results: {len(resolved_ambiguous):,}")

In [ ]:
if not MANUAL_FILE.exists():
    manual_template = pd.DataFrame(
        columns=[
            "summons_number",
            "recovered_borough",
        ]
    )

    manual_template.to_csv(MANUAL_FILE, index=False)
    print(f"Created manual-review template:\n{MANUAL_FILE}")

In [ ]:
manual_results = pd.read_csv(
    MANUAL_FILE,
    dtype="string",
)

manual_results = manual_results[
    manual_results["recovered_borough"].isin(VALID_BOROUGHS)
].copy()

manual_results["confidence"] = "manual"
manual_results["recovery_method"] = "manual_review"
manual_results["source_file"] = MANUAL_FILE.name

manual_results = manual_results[
    [
        "summons_number",
        "recovered_borough",
        "confidence",
        "recovery_method",
        "source_file",
    ]
]

print(f"Manual results: {len(manual_results):,}")
manual_results.head()

In [ ]:
recovery = pd.concat(
    [
        accepted_results,
        resolved_ambiguous,
        manual_results,
    ],
    ignore_index=True,
)

recovery["summons_number"] = pd.to_numeric(
    recovery["summons_number"],
    errors="coerce",
).astype("Int64")

recovery = recovery.dropna(subset=["summons_number", "recovered_borough"])

# Check for conflicting boroughs assigned to the same summons
conflicts = recovery.groupby("summons_number")["recovered_borough"].nunique()

conflicts = conflicts[conflicts > 1]

if not conflicts.empty:
    conflicting_rows = recovery[
        recovery["summons_number"].isin(conflicts.index)
    ].sort_values("summons_number")

    display(conflicting_rows)

    raise ValueError(f"{len(conflicts):,} summons numbers have conflicting boroughs")

# Manual results were added last, so they are retained for duplicates
recovery = recovery.drop_duplicates(
    subset=["summons_number"],
    keep="last",
)

recovery["summons_number"] = recovery["summons_number"].astype("int64")

print(f"Total approved recovery results: {len(recovery):,}")

recovery["recovered_borough"].value_counts()

In [ ]:
recovery.to_csv(
    RECOVERY_FILE,
    index=False,
)

print(f"Saved {len(recovery):,} rows to:")
print(RECOVERY_FILE)

In [ ]:
records = [
    (
        int(row.summons_number),
        row.recovered_borough,
        None if pd.isna(row.confidence) else str(row.confidence),
        None if pd.isna(row.recovery_method) else str(row.recovery_method),
        None if pd.isna(row.source_file) else str(row.source_file),
    )
    for row in recovery.itertuples(index=False)
]

with sqlite3.connect(DATABASE_FILE) as connection:
    connection.execute("PRAGMA foreign_keys = ON")

    connection.executescript("""
        CREATE TABLE IF NOT EXISTS borough_recovery (
            summons_number INTEGER PRIMARY KEY,
            recovered_borough TEXT NOT NULL,
            confidence TEXT,
            recovery_method TEXT,
            source_file TEXT,
            FOREIGN KEY (summons_number)
                REFERENCES parking_violations(summons_number)
        );

        DELETE FROM borough_recovery;

        DROP TABLE IF EXISTS temp.recovery_stage;

        CREATE TEMP TABLE recovery_stage (
            summons_number INTEGER PRIMARY KEY,
            recovered_borough TEXT,
            confidence TEXT,
            recovery_method TEXT,
            source_file TEXT
        );
        """)

    connection.executemany(
        """
        INSERT INTO recovery_stage
        VALUES (?, ?, ?, ?, ?)
        """,
        records,
    )

    # Only insert summons numbers found in parking_violations
    connection.execute("""
        INSERT INTO borough_recovery
        SELECT
            s.summons_number,
            s.recovered_borough,
            s.confidence,
            s.recovery_method,
            s.source_file
        FROM recovery_stage AS s
        INNER JOIN parking_violations AS p
            ON p.summons_number = s.summons_number
        """)

    loaded_count = connection.execute(
        "SELECT COUNT(*) FROM borough_recovery"
    ).fetchone()[0]

print(f"Loaded {loaded_count:,} recovered boroughs into SQLite.")

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    connection.executescript("""
        DROP VIEW IF EXISTS parking_analysis;

        CREATE VIEW parking_analysis AS
        SELECT
            p.*,

            COALESCE(
                NULLIF(TRIM(p.borough), ''),
                r.recovered_borough
            ) AS analysis_borough,

            CASE
                WHEN NULLIF(TRIM(p.borough), '') IS NOT NULL
                    THEN 'original'
                WHEN r.recovered_borough IS NOT NULL
                    THEN 'recovered'
                ELSE 'missing'
            END AS borough_source,

            r.confidence AS recovery_confidence,
            r.recovery_method,
            r.source_file AS recovery_source_file

        FROM parking_enriched AS p

        LEFT JOIN borough_recovery AS r
            ON r.summons_number = p.summons_number;
        """)

print("Created SQLite view: parking_analysis")

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    validation = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_summons,

            SUM(
                CASE WHEN borough_source = 'original'
                THEN 1 ELSE 0 END
            ) AS original_boroughs,

            SUM(
                CASE WHEN borough_source = 'recovered'
                THEN 1 ELSE 0 END
            ) AS recovered_boroughs,

            SUM(
                CASE WHEN borough_source = 'missing'
                THEN 1 ELSE 0 END
            ) AS still_missing

        FROM parking_analysis
        """,
        connection,
    )

validation

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    borough_analysis = pd.read_sql_query(
        """
        SELECT
            analysis_borough AS borough,
            borough_source,
            COUNT(*) AS summons_count
        FROM parking_analysis
        GROUP BY
            analysis_borough,
            borough_source
        ORDER BY summons_count DESC
        """,
        connection,
    )

borough_analysis

In [ ]:
""" camera_results = pd.read_csv(
    PROJECT_ROOT / "data/processed/geosupport_camera_description_accepted.csv",
    dtype="string",
)

camera_results = camera_results[
    [
        "summons_number",
        "suggested_borough",
        "confidence",
        "resolution_method",
    ]
].rename(
    columns={
        "suggested_borough": "recovered_borough",
        "resolution_method": "recovery_method",
    }
)

camera_results["source_file"] = "geosupport_camera_description_accepted.csv"

recovery = pd.concat(
    [
        accepted_results,
        resolved_ambiguous,
        manual_results,
        camera_results,
    ],
    ignore_index=True,
) """